# Sparse Circuit from a Netlist

This example simulates a pulse-driven LC low-pass filter with a resistive load using the `NetlistCircuit` block. [ElecSolver](https://github.com/williampiat3/ElecSolver) parses the SPICE-like netlist; PathSim then integrates the circuit as a sparse linear differential-algebraic equation (DAE).

The block behaves like a state space model of the netlist:

- the **inputs** are the independent sources, named like in the netlist (`V1`, `Bload`, ...)
- the **outputs** are all node voltages `V(node)` and all element currents `I(name)`

so the circuit is wired to other blocks by name.

**Before running:** select a Python environment with `pathsim`, `pathsim_rf`, `matplotlib`, and the optional dependency `ElecSolver` (`pip install ElecSolver`). If you use local source checkouts, keep their directories on `PYTHONPATH`. Run the cells from top to bottom: inspect the available ports, select the measurements, connect the blocks, and run the simulation.

In [ ]:
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import PulseSource, Amplifier, Scope
from pathsim.solvers import BDF2
from pathsim_rf import NetlistCircuit

## Netlist

The source `V1` drives node `in` through a 0.5 mH series inductor (`Lfilter`). A 50 µF capacitor (`Cfilter`) connects the output node `n1` to ground, node `0`.

The 50 Ω load is represented by a behavioral current source, `Bload`. Its expression `I=V(n1)/50` describes the intended load, but the block does **not** evaluate it: the current is an input supplied by the PathSim feedback connection below.

Likewise, the `PULSE(...)` waveform and `.tran` directive do not control the PathSim run. We set the matching pulse parameters in `PulseSource` and the timestep and duration in the simulation cells.

In [ ]:
netlist = """
*validation netlist for pathsim
.param FREQ = 9k
.param DUTY = 0.5
.param V_LOW = 0
.param V_HIGH = 1
.param T_PER = {1/FREQ}
.param T_ON = {T_PER * DUTY}
.param T_RISE = 10n
.param T_FALL = 10n

V1 in 0 PULSE({V_LOW} {V_HIGH} 0 {T_RISE} {T_FALL} {T_ON} {T_PER})
Bload n1 0 I=V(n1)/50
Lfilter in n1 0.5e-3
Cfilter n1 0 50e-6
.tran 0.5e-6 40e-3 0 0.00001
.end
"""

## Circuit block

First, create a block with all outputs exposed and print the available port names. Inputs list voltage sources first, then current sources. Outputs list node voltages relative to ground (ground itself is excluded), then element currents. A current is positive from the first node to the second node in that element's netlist declaration.

The default initial state is zero: the capacitor is uncharged and the inductor carries no current. You can set these quantities by name with `initial_value`, for example `{"V(n1)": 1.0, "I(Lfilter)": 0.5}` (volts and amperes).

In [ ]:
circuit = NetlistCircuit(netlist)

print("inputs :", circuit.input_names)
print("outputs:", circuit.output_names)

Now replace the inspection block with one exposing only the two measurements used below: `V(n1)` for the load feedback and plot, and `I(Lfilter)` for the plot. The selection lists take bare node and element names, not the formatted port labels.

This reduces the rows of the sparse output matrices `C` and `D` inherited from `SparseLinearDAE`; it does not remove components or reduce the internal circuit state. Selecting outputs without direct feedthrough can also avoid unnecessary algebraic dependencies, but output selection alone does not guarantee that a feedback loop disappears.

In [ ]:
circuit = NetlistCircuit(netlist, output_voltages=["n1"], output_currents=["Lfilter"])

print("inputs :", circuit.input_names)
print("outputs:", circuit.output_names)

## Blocks and connections

The load feedback is implemented outside the circuit: `V(n1)` passes through a gain of `1/50` A/V and returns to the input `Bload`. This implements Ohm's law, drawing current from `n1` to ground just like a 50 Ω resistor.

`PulseSource` drives `V1` with a 0-to-1 V pulse at 9 kHz and 50% duty cycle, matching the netlist. The scope records the source voltage, filtered output voltage, and inductor current. Named circuit ports make these connections independent of their numeric port indices.

In [ ]:
Src = PulseSource(T=1/9000, amplitude=1, duty=0.5, t_rise=10e-9, t_fall=10e-9)
Load = Amplifier(1/50)
Sco = Scope(labels=["pulse V1", "V(n1)", "I(Lfilter)"])

blocks = [Src, circuit, Load, Sco]

connections = [
    Connection(Src, circuit["V1"], Sco[0]),
    Connection(circuit["V(n1)"], Load, Sco[1]),
    Connection(Load, circuit["Bload"]),
    Connection(circuit["I(Lfilter)"], Sco[2]),
    ]

## Simulation

Use the implicit BDF2 solver to integrate the circuit DAE, with an initial timestep of 0.5 µs. The simulation starts from the zero initial state described above; `log=True` enables progress logging. The next cell constructs the simulation; the final cell advances it for 40 ms.

In [ ]:
Sim = Simulation(blocks, connections, Solver=BDF2, dt=0.5e-6, log=True)

## Results

Run the simulation and plot the three recorded signals. `pulse V1` and `V(n1)` are in volts, while `I(Lfilter)` is in amperes, so their plotted magnitudes have different units.

Look for the LC startup oscillation and the smoothing of the input pulses at `V(n1)`. With a 0-to-1 V input and 50% duty cycle, the long-term mean output should be near 0.5 V; zoom in on the end of the trace to inspect the remaining switching ripple.

In [ ]:
Sim.run(40e-3)
Sco.plot(lw=1.5)
plt.show()